In [ ]:
# ── Filter Responsibility ─────────────────────────────────────────────────────
# The following filter from the spec is intentionally NOT applied in this notebook.
# It is applied as a Power BI page-level filter instead:
#
#   gl_date BETWEEN '2026-05-01' AND '2026-05-31'   → SDDGL
#
# The Gold table stores ALL invoice-line rows regardless of GL date.
# Power BI page-level filter narrows to the current reconciliation month.
# ─────────────────────────────────────────────────────────────────────────────

In [1]:
# In[1]:

import threading
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

GOLD_SCHEMA = "lh_jde_gold.rpt"

ENV             = "dev"                              # checkpoint namespacing — envs never collide
TRIGGER         = {"processingTime": "30 seconds"}   # ← continuous; refresh every 30 s
CKPT            = f"Files/checkpoints/eso4_fact_{ENV}"  # OWN root — independent of the dim notebooks

#   OVERWRITE = True  -> full load: drop + rebuild the fact from the full Silver snapshot,
#                        snapshot each streamed source's Delta version as init_ver, clear checkpoints.
#   OVERWRITE = False -> resume: keep the table + checkpoints, streams catch up from where
#                        they left off (init_ver = -1, no version filtering).
OVERWRITE       = True    # ⚠ ONE-OFF full reprocess — set back to False after a healthy run

# Serialises all fact-table writes: foreachBatch handlers run in separate driver threads, so
# this lock makes the F4211 and F03B11 streams take turns (delete+append on the same Gold fact
# never overlaps). 
_FACT_LOCK      = threading.Lock()

# ── Silver sources ────────────────────────────────────────────────────────────
SRC_SCHEMA    = "jde_cdc"     # Silver Change Data Feed schema; CDF must be enabled on every
                          #   STREAMED source (F4211 / F03B11) read below.
SRC_LAKEHOUSE = "lh_jde_silver"
F4211_TBL   = "f4211_sales_order_detail_file"      # streamed
F03B11_TBL  = "f03b11_customer_ledger"             # streamed
F0006_TBL   = "f0006_business_unit_master"         # static snapshot (business_stream calc input MCRP20)
F0101_TBL   = "f0101_address_book_master"          # static snapshot (sic_code FK)
F0116_TBL   = "f0116_address_by_date"              # static snapshot (jurisdiction FK / county)
# F0005 (UDC) is NOT read here — dim_sic (01/SC) + dim_state (00/S) are built by nb_eso4_gold_dim_udc.py.

# ── Gold target BUILT here (new, eso4) ─────────────────────────────────────────
T_FACT      = f"{GOLD_SCHEMA}.fact_sales_tax_reconciliation"

# ── report scaling ─────────────────────────────────────────────────────────────
# Hubble multiplied the four amounts by NVL(company.ShiftFactor, 0.01) to de-scale RAW JDE
# integer amounts (stored ×100). Our Silver is already decoded (implied decimals resolved), so
# the placeholder is 1.0. `shift_factor_applied` is
# carried on the fact for lineage. (See design §"ShiftFactor".)
SHIFT_FACTOR    = 1.0

print(f"ESO4 Gold fact processor — trigger {TRIGGER}  target {T_FACT}")

StatementMeta(, fd5a786b-186b-43b4-8918-1a88fade26af, 3, Finished, Available, Finished, False)

ESO4 Gold fact processor — trigger {'processingTime': '30 seconds'}  target lh_jde_gold.rpt.fact_sales_tax_reconciliation


In [2]:
# In[2]:

_SOFT_DELETE_COLS = ["is_delete", "deleted_date_time"]

def sname(table_name):
    """Fully-qualified Silver source name."""
    return f"{SRC_LAKEHOUSE}.{SRC_SCHEMA}.{table_name}"

def load_silver_table(table_name):
    df = spark.table(sname(table_name))
    if "is_delete" in df.columns:
        df = df.filter(F.col("is_delete") == 0)
    return df.select(*[c for c in df.columns if c not in _SOFT_DELETE_COLS])

def sk(*cols):
    return F.sha2(F.concat_ws("||", *[F.col(c).cast("string") if isinstance(c, str) else c.cast("string")
                                       for c in cols]), 256)

def current_version(silver_table):
    """Latest committed Delta version of a Silver source (for init_ver seed-skip)."""
    return spark.sql(f"DESCRIBE HISTORY {sname(silver_table)}").select(F.max("version")).first()[0]

StatementMeta(, fd5a786b-186b-43b4-8918-1a88fade26af, 4, Finished, Available, Finished, False)

In [3]:
# In[3]:

def _write_new_table(df, target, cdf=True):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
    w = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if cdf:
        w = w.option("delta.enableChangeDataFeed", "true")   # Gold CDF on for downstream
    w.saveAsTable(target)

def recompute_fact(docs):
    """CDC for the fact: delete the affected invoice-document scope then append the recomputed
    lines. `docs` = distinct
    company_key / document_type / invoice_number. Returns rows written."""
    if docs.rdd.isEmpty():
        return 0
    src = transform_fact(restrict_docs=docs)
    with _FACT_LOCK:
        if not spark.catalog.tableExists(T_FACT):
            _write_new_table(src, T_FACT)
            return src.count()
        scope = docs.select(sk("company_key", "document_type", "invoice_number")
                            .alias("document_scope_key")).distinct()
        (DeltaTable.forName(spark, T_FACT).alias("t")
            .merge(scope.alias("s"), "t.document_scope_key = s.document_scope_key")
            .whenMatchedDelete().execute())               # drop the document's old lines
        src.write.format("delta").mode("append").saveAsTable(T_FACT)   # append current lines
    return src.count()

StatementMeta(, fd5a786b-186b-43b4-8918-1a88fade26af, 5, Finished, Available, Finished, False)

In [4]:
# In[4]:

FACT_GROUP_BY_COLS = [
    # ── degenerate / document identifiers ──
    "document_company",        # XF4211_SDKCO (= XF03B11_RPOKCO)
    "invoice_number",          # XF4211_SDDOC (= XF03B11_RPODOC)
    "document_type",           # XF4211_SDDCT (= XF03B11_RPODCT)
    "order_number",            # XF4211_SDDOCO
    "order_type",              # XF4211_SDDCTO
    # ── dimension FKs ──
    "plant",                   # XF4211_SDMCU (= XC_F4211_SDMCU)
    "ship_to",                 # XF4211_SDSHAN
    "sold_to",                 # XF4211_SDAN8
    "parent_number",           # XF4211_SDPA8
    # ── tax attributes (degenerate) ──
    "tax_explanation_code",    # XF4211_SDEXR1
    "tax_area",                # XF03B11_RPTXA1
    "avalara_code",            # XID_CUSTOM_8501fecff7aa51 (calc)
    # ── classification (calc + denormalized slicer attrs) ──
    "business_stream",         # XID_CUSTOM_84c76537583683 (calc; degenerate — needs ABSIC×MCRP20)
    "sic_code",                # XF0101_ABSIC — FK → dim_sic (description resolved in the model)
    # ── address / FK codes ──
    "jurisdiction",            # XF0116_ALADDS raw code (e.g. "CO") — FK → dim_state (name in dim)
    "county",                  # XF0116_ALCOUN (degenerate — not in the reused address dim)
    # ── raw event dates ──
    "gl_date",                 # XF4211_SDDGL
    "service_tax_date",        # XF03B11_RPDSVJ
]
# Attributes functionally dependent on the grain (carried through the aggregation, not grouped on).
# Full star schema: plant_name / business_stream_code (dim_business_unit), sic_description (dim_sic),
# and the state name (dim_state) all live in dimensions now — only the constant remains on the fact.
# business_stream_code is dropped from the GROUP BY too (functionally dependent on plant → dim_business_unit).
FACT_CARRY_COLS = ["shift_factor_applied"]
# SUMmed measures (Hubble ReportColumn1-4).
FACT_MEASURE_COLS = ["taxable_amount", "non_taxable_amount", "tax_amount", "gross_amount"]
# Stored fact columns, in report order — degenerate dims + FK codes + measures ONLY (star schema).
FACT_BUSINESS_COLS = [
    "document_company", "invoice_number", "document_type", "order_number", "order_type",
    "plant", "ship_to", "sold_to", "parent_number",   # FKs → dim_business_unit / dim_address_*
    "tax_explanation_code", "tax_area", "avalara_code",
    "business_stream", "sic_code",                    # business_stream calc (degenerate); sic_code FK → dim_sic
    "jurisdiction", "county",                         # jurisdiction FK → dim_state; county degenerate
    "gl_date", "service_tax_date",
    "taxable_amount", "non_taxable_amount", "tax_amount", "gross_amount", "shift_factor_applied",
]

def transform_fact(restrict_docs=None):
    sd  = load_silver_table(F4211_TBL)       # sales order detail (streamed source)
    ar  = load_silver_table(F03B11_TBL)      # customer ledger    (streamed source)
    bu  = load_silver_table(F0006_TBL)       # business unit master (static; business_stream calc input)
    ab  = load_silver_table(F0101_TBL)       # address book master  (static; sic_code)
    adr = load_silver_table(F0116_TBL)       # address by date      (static; jurisdiction/county)

    # scope restriction (CDC): keep only the changed invoice documents' F4211 lines
    if restrict_docs is not None:
        sd = sd.join(restrict_docs.alias("rd"),
                     (sd["company_key"] == F.col("rd.company_key")) &
                     (sd["document_type"] == F.col("rd.document_type")) &
                     (sd["doc_voucher_invoice_e"] == F.col("rd.invoice_number")), "left_semi")
                     
    address = (ab.alias("ab")
               .join(adr.alias("ad"), F.col("ab.address_number") == F.col("ad.address_number"), "inner")
               .groupBy(F.col("ab.address_number").alias("addr_key"))
               .agg(F.first(F.trim(F.col("ab.standard_industry_code")), ignorenulls=True).alias("sic_code"),   # ABSIC (FK dim_sic)
                    F.first(F.trim(F.col("ad.state")),                   ignorenulls=True).alias("jurisdiction"),  # ALADDS (FK dim_state)
                    F.first(F.trim(F.col("ad.county_address")),          ignorenulls=True).alias("county")))       # ALCOUN

    # F0006 collapsed to one row per business unit — business_stream_code (MCRP20) feeds the
    # Business Stream calc only; plant_name lives in dim_business_unit (not carried on the fact).
    bunit = (bu.groupBy(F.col("cost_center").alias("bu_key"))
             .agg(F.first(F.trim(F.col("category_code_cost_ct_020")), ignorenulls=True).alias("business_stream_code")))  # MCRP20

    # ── joins (docx §4, every join used; NO filters) ──────────────────────────────
    j = (sd.alias("sd")
         .join(ar.alias("ar"),                                                     # INNER
               (F.col("sd.doc_voucher_invoice_e") == F.col("ar.original_document_no")) &   # SDDOC = RPODOC
               (F.col("sd.document_type")         == F.col("ar.original_document_type")) & # SDDCT = RPODCT
               (F.col("sd.company_key")           == F.col("ar.company_key_original")) &   # SDKCO = RPOKCO
               (F.col("sd.line_number")           == F.col("ar.line_number")), "inner")    # SDLNID = RPLNID
         .join(bunit.alias("bu"), F.col("sd.cost_center") == F.col("bu.bu_key"), "left")   # SDMCU = MCMCU
         .join(address.alias("ad"), F.col("sd.address_number_ship_to") == F.col("ad.addr_key"), "left"))  # SDSHAN = ABAN8

    # ── Business Stream (docx §7 calculation) — ABSIC (F0101) × MCRP20 (F0006) ──────
    _absic  = F.trim(F.col("ad.sic_code"))
    _mcrp20 = F.trim(F.col("bu.business_stream_code"))
    business_stream = (F.when((_absic == "F") & (_mcrp20 == "ENG"), F.lit("O&G"))
                        .when((_absic != "F") & (_mcrp20 == "ENG"), F.lit("ISP"))
                        .when((_absic != "F") & (_mcrp20 == "SHR"), F.lit("ISP"))
                        .when((_absic == "F") & (_mcrp20 == "SHR"), F.lit("O&G"))
                        .when(~_mcrp20.isin("ENG", "SHR"), F.lit("ISP")))


    avalara_code = F.concat(
        F.coalesce(F.trim(F.col("sd.doc_voucher_invoice_e").cast("long").cast("string")), F.lit("-999999999")),
        F.coalesce(F.trim(F.col("sd.document_type")),                                      F.lit("")),
        F.coalesce(F.trim(F.col("sd.company_key").cast("string")),                         F.lit("")))

    sel = j.select(
        # ── degenerate / document identifiers ──
        F.col("sd.company_key").alias("document_company"),               # SDKCO ("Document Company")
        F.col("sd.doc_voucher_invoice_e").alias("invoice_number"),       # SDDOC
        F.col("sd.document_type").alias("document_type"),                # SDDCT
        F.col("sd.document_order_invoice_e").alias("order_number"),      # SDDOCO
        F.col("sd.order_type").alias("order_type"),                      # SDDCTO
        F.col("ar.doc_voucher_invoice_e").alias("ar_document_no"),       # RPDOC  (F03B11 PK)
        F.col("ar.document_type").alias("ar_document_type"),             # RPDCT  (F03B11 PK)
        F.col("ar.company_key").alias("ar_company_key"),                 # RPKCO  (F03B11 PK)
        F.col("ar.document_pay_item").alias("ar_pay_item"),              # RPSFX  (F03B11 PK)
        # ── dimension FKs ──
        F.trim(F.col("sd.cost_center")).alias("plant"),                  # SDMCU -> dim_business_unit
        F.col("sd.address_number").alias("sold_to"),                     # SDAN8  (docx col 19 "Sold To")
        F.col("sd.address_number_ship_to").alias("ship_to"),             # SDSHAN (docx col 20 "Ship To")
        F.col("sd.address_number_parent").alias("parent_number"),        # SDPA8
        # ── tax attributes ──
        F.col("sd.tax_explanation_code_01").alias("tax_explanation_code"),  # SDEXR1
        F.col("ar.tax_area_01").alias("tax_area"),                          # RPTXA1
        avalara_code.alias("avalara_code"),
        # ── classification (FK codes; descriptions resolved in the model) ──
        business_stream.alias("business_stream"),                        # calc (§7) — degenerate
        F.col("ad.sic_code").alias("sic_code"),                          # ABSIC (F0101) — FK → dim_sic
        # ── address FK code + degenerate county ──
        F.col("ad.state").alias("jurisdiction"),                  # ALADDS (F0116) raw code — FK → dim_state
        F.col("ad.county").alias("county"),                              # ALCOUN (F0116) — degenerate
        # ── raw event dates ──
        F.col("sd.dt_for_gl_and_vouch_01").alias("gl_date"),             # SDDGL
        F.col("ar.date_service_currency").alias("service_tax_date"),     # RPDSVJ
        # ── measures (× SHIFT_FACTOR) ──
        (F.col("ar.amount_taxable")    * F.lit(SHIFT_FACTOR)).alias("taxable_amount"),      # RPATXA
        (F.col("ar.amount_tax_exempt") * F.lit(SHIFT_FACTOR)).alias("non_taxable_amount"),  # RPATXN
        (F.col("ar.amt_tax_02")        * F.lit(SHIFT_FACTOR)).alias("tax_amount"),          # RPSTAM
        (F.col("ar.amount_gross")      * F.lit(SHIFT_FACTOR)).alias("gross_amount"),        # RPAG
        F.lit(SHIFT_FACTOR).cast("double").alias("shift_factor_applied"),
    ).distinct()                                            # == Hubble inner SELECT DISTINCT (incl F03B11 PK)

    agg = (sel.groupBy(*FACT_GROUP_BY_COLS)
           .agg(F.sum("taxable_amount").alias("taxable_amount"),           # SUM ReportColumn1
                F.sum("non_taxable_amount").alias("non_taxable_amount"),   # SUM ReportColumn2
                F.sum("tax_amount").alias("tax_amount"),                   # SUM ReportColumn3
                F.sum("gross_amount").alias("gross_amount"),               # SUM ReportColumn4
                F.first("shift_factor_applied").alias("shift_factor_applied")))         # constant

    df = (agg
          # delete-scope key (invoice document) + unique key at the GROUP BY grain
          .withColumn("document_scope_key",
                      sk("document_company", "document_type", "invoice_number"))
          .withColumn("sales_tax_line_key", sk(*FACT_GROUP_BY_COLS)))

    df = df.dropDuplicates(["sales_tax_line_key"])         # unique by construction (GROUP BY); defensive
    return df.select("sales_tax_line_key", "document_scope_key", *FACT_BUSINESS_COLS)

StatementMeta(, fd5a786b-186b-43b4-8918-1a88fade26af, 6, Finished, Available, Finished, False)

In [5]:
# In[5]:


_CKPT_PATHS = [f"{CKPT}/fact__{F4211_TBL}", f"{CKPT}/fact__{F03B11_TBL}"]

def _checkpoints_exist():
    """True iff EVERY per-stream checkpoint has a COMMITTED offset (its offsets/ dir is
    non-empty). A checkpoint dir can hold metadata/ + sources/ yet never have committed a
    batch; requiring offsets/ treats such an INCOMPLETE checkpoint as ABSENT and forces a
    FULL LOAD (re-establishing init_ver) rather than cold-starting the CDF reader at
    startingVersion=0 (version 0 predates CDF enablement -> DELTA_MISSING_CHANGE_DATA)."""
    for p in _CKPT_PATHS:
        try:
            if not mssparkutils.fs.ls(f"{p}/offsets"):
                return False
        except Exception:
            return False
    return True

# ── (1) stop leftover streams from a previous run in this Spark session ──────────
_STREAM_NAMES = {"fact__" + F4211_TBL, "fact__" + F03B11_TBL}
_stopped = []
for _q in list(spark.streams.active):
    if _q.name in _STREAM_NAMES:
        _q.stop()
        _stopped.append(_q.name)
if _stopped:
    print(f"Stopped leftover streams: {_stopped}")

# ── (2/3) full-load gate — also full-load when the checkpoints are missing/incomplete ──
_FULL_LOAD = OVERWRITE or not spark.catalog.tableExists(T_FACT) or not _checkpoints_exist()

if _FULL_LOAD:
    print("== FULL LOAD ==")
    spark.sql(f"DROP TABLE IF EXISTS {T_FACT}")
    _write_new_table(transform_fact(), T_FACT)
    print(f"  ✓ seeded {T_FACT}")
    _init_ver = {t: current_version(t) for t in (F4211_TBL, F03B11_TBL)}
    print(f"  init versions: {_init_ver}")
    try:
        mssparkutils.fs.rm(CKPT, True)
        print("  checkpoints cleared")
    except Exception as e:
        print(f"  checkpoint clear skipped: {e}")
    print("✓ full load complete")
else:
    print("== RESUME from checkpoint ==")
    _init_ver = {}   # .get(src, -1) -> -1 everywhere; no version filtering, checkpoint drives

StatementMeta(, fd5a786b-186b-43b4-8918-1a88fade26af, 7, Finished, Available, Finished, False)

== FULL LOAD ==
  ✓ seeded lh_jde_gold.rpt.fact_sales_tax_reconciliation
  init versions: {'f4211_sales_order_detail_file': 1244, 'f03b11_customer_ledger': 15}
  checkpoints cleared
✓ full load complete


In [6]:
# In[6]:


def make_fact_f4211_handler(init_ver):
    def handler(batch_df, batch_id):
        if batch_df.rdd.isEmpty():
            return
        if init_ver >= 0:
            batch_df = batch_df.filter(F.col("_commit_version") > init_ver)
        if batch_df.rdd.isEmpty():
            return
        docs = (batch_df.filter(F.col("_change_type").isin("insert", "update_postimage", "delete"))
                .select(F.col("company_key"), F.col("document_type"),
                        F.col("doc_voucher_invoice_e").alias("invoice_number")).distinct())
        n = recompute_fact(docs)   # delete document scope + append recomputed lines (self-locks)
        print(f"[{F4211_TBL[:12]}] fact batch={batch_id} rows={n}")
    return handler

def make_fact_f03b11_handler(init_ver):
    def handler(batch_df, batch_id):
        if batch_df.rdd.isEmpty():
            return
        if init_ver >= 0:
            batch_df = batch_df.filter(F.col("_commit_version") > init_ver)
        if batch_df.rdd.isEmpty():
            return
        # F03B11 change rows point back to the F4211 invoice via the ORIGINAL-document keys
        # (RPOKCO/RPODCT/RPODOC == SDKCO/SDDCT/SDDOC) — recompute those documents' lines.
        docs = (batch_df.filter(F.col("_change_type").isin("insert", "update_postimage", "delete"))
                .select(F.col("company_key_original").alias("company_key"),
                        F.col("original_document_type").alias("document_type"),
                        F.col("original_document_no").alias("invoice_number"))
                .where(F.col("invoice_number").isNotNull()).distinct())
        n = recompute_fact(docs)
        print(f"[{F03B11_TBL[:12]}] fact batch={batch_id} rows={n}")
    return handler

StatementMeta(, fd5a786b-186b-43b4-8918-1a88fade26af, 8, Finished, Available, Finished, False)

In [7]:
# In[7]:


def _start_ver(iv, tbl):
    """Full load: init_ver (exists, carries CDF; handler skips <= it). Resume (iv < 0):
    fall back to the source's CURRENT version, never 0 (v0 predates CDF enablement and would
    raise DELTA_MISSING_CHANGE_DATA); on a genuine resume the committed offset drives anyway."""
    return iv if iv >= 0 else current_version(tbl)

iv_fact = _init_ver.get(F4211_TBL, -1)
_sv_fact = _start_ver(iv_fact, F4211_TBL)
(spark.readStream.format("delta")
     .option("readChangeFeed",  "true")
     .option("startingVersion", _sv_fact)
     .table(sname(F4211_TBL))
 .writeStream
     .foreachBatch(make_fact_f4211_handler(iv_fact))
     .option("checkpointLocation", f"{CKPT}/fact__{F4211_TBL}")
     .trigger(**TRIGGER)
     .queryName("fact__" + F4211_TBL)
     .start())
print(f"  fact__{F4211_TBL}  startingVersion={_sv_fact}  init_ver={iv_fact}")

iv_ar = _init_ver.get(F03B11_TBL, -1)
_sv_ar = _start_ver(iv_ar, F03B11_TBL)
(spark.readStream.format("delta")
     .option("readChangeFeed",  "true")
     .option("startingVersion", _sv_ar)
     .table(sname(F03B11_TBL))
 .writeStream
     .foreachBatch(make_fact_f03b11_handler(iv_ar))
     .option("checkpointLocation", f"{CKPT}/fact__{F03B11_TBL}")
     .trigger(**TRIGGER)
     .queryName("fact__" + F03B11_TBL)
     .start())
print(f"  fact__{F03B11_TBL}  startingVersion={_sv_ar}  init_ver={iv_ar}")

print(f"== started 2 streams — continuous, trigger {TRIGGER}. Target {T_FACT}. "
      "dim_business_unit refreshes via its own job; dim_address reused (rpt). ==")
spark.streams.awaitAnyTermination()

StatementMeta(, fd5a786b-186b-43b4-8918-1a88fade26af, 9, Finished, Available, Finished, True)

  fact__f4211_sales_order_detail_file  startingVersion=1244  init_ver=1244
  fact__f03b11_customer_ledger  startingVersion=15  init_ver=15
== started 2 streams — continuous, trigger {'processingTime': '30 seconds'}. Target lh_jde_gold.rpt.fact_sales_tax_reconciliation. dim_business_unit refreshes via its own job; dim_address reused (rpt). ==


StreamingQueryException: [STREAM_FAILED] Query [id = f0d557f5-c96d-48d9-b410-b4e344eb3598, runId = b3944486-ba3c-4dda-9588-abb90ceece1f] terminated with exception: [DELTA_MISSING_CHANGE_DATA] Error getting change data for range [15 , 16] as change data was not
recorded for version [15]. If you've enabled change data feed on this table,
use `DESCRIBE HISTORY` to see when it was first enabled.
Otherwise, to start recording change data, use `ALTER TABLE table_name SET TBLPROPERTIES
(delta.enableChangeDataFeed=true)`.